In [6]:
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from glob import glob
import subprocess
import matplotlib.pyplot as plt
import os

os.chdir('/home/vladimirnoz/Projects/Codebook_Perspectives/AS_CHS_GHTS/')

In [3]:
def merge_intervals(input_path, output_path):
    files = glob(f'{input_path}/*.bed')
    files_as_str = ' '.join(files)
    cmd = f'cat {files_as_str} | bedtools sort | bedtools merge > {output_path}'
    print(subprocess.run(cmd, shell=True, capture_output=True, text=True).stderr)

In [10]:
merge_intervals('/home/vladimirnoz/Projects/Codebook_Perspectives/TOPs/TOPs', '/home/vladimirnoz/Projects/Codebook_Perspectives/TOPs/merged.bed')
merge_intervals('/home/vladimirnoz/Projects/Codebook_Perspectives/peaks/chipseq+ghtselex', '/home/vladimirnoz/Projects/Codebook_Perspectives/peaks/merged.bed')

In [7]:
def merge_ase_thr(input_path, output_path, thr=1):
    files = glob(input_path)
    tables = []
    for file in tqdm(files):
        df = pd.read_table(file)
        mask = df['fdr_comb_pval'] < thr
        tables.append(df[mask])
    table = pd.concat(tables, ignore_index=True)
    table_text = table.to_csv(sep='\t', index=False, header=False)
    cmd = f'bedtools sort | bedtools merge > {output_path}'
    print(subprocess.run(cmd, shell=True, capture_output=True, text=True, input=table_text).stderr)

In [9]:
merge_ase_thr('as_tables/*/ASB_motifs/*.tsv', '/tmp/ase_merged_nothr.bed', thr=1)
merge_ase_thr('as_tables/*/ASB_motifs/*.tsv', '/tmp/ase_merged_thr005.bed', thr=0.05)

  0%|          | 0/333 [00:00<?, ?it/s]

  0%|          | 0/333 [00:00<?, ?it/s]

In [13]:
def intersect_and_count(interval, ase_nonsignif, ase_signif):
    interval_len_cmd = f"cat {interval} | "  + "awk -F'\t' 'BEGIN{SUM=0}{ SUM+=$3-$2 }END{print SUM}'"
    interval_len = subprocess.run(interval_len_cmd, shell=True, text=True, capture_output=True)
    interval_len = int(interval_len.stdout.strip())
    
    total_ase_cmd = f'cat {ase_nonsignif} | wc -l'
    total_ase_count = subprocess.run(total_ase_cmd, shell=True, text=True, capture_output=True)
    total_ase_count = int(total_ase_count.stdout.strip())
    
    intersected_ase_cmd = f'bedtools intersect -a {ase_nonsignif} -b {interval} | wc -l'
    intersected_ase_count = subprocess.run(intersected_ase_cmd, shell=True, text=True, capture_output=True)
    intersected_ase_count = int(intersected_ase_count.stdout.strip())

    signif_cmd = f'bedtools intersect -a {ase_nonsignif} -b {interval} | \
                   bedtools intersect -a stdin -b {ase_signif} | wc -l'
    signif_count = subprocess.run(signif_cmd, shell=True, text=True, capture_output=True)
    signif_count = int(signif_count.stdout.strip())
    return [interval_len, total_ase_count, intersected_ase_count, signif_count]



In [14]:
intervals = ['/home/vladimirnoz/Projects/Codebook_Perspectives/peaks/merged.bed', '/home/vladimirnoz/Projects/Codebook_Perspectives/TOPs/merged.bed']

In [17]:
from itertools import product

table = []
for inter in intervals:
    row = intersect_and_count(inter, '/tmp/ase_merged_nothr.bed', '/tmp/ase_merged_thr005.bed')
    row = [inter.split('/')[-2], *row]
    table.append(row)
df = pd.DataFrame(table, columns=['intervals', 'intervals_len', 'total_snp_count', 'interval_snp_count', 'asb_count'])
df

,intervals,intervals_len,total_snp_count,interval_snp_count,asb_count
0,peaks,2359071386,121611,121019,9973
1,TOPs,12085440,121611,3091,945


In [18]:
df = pd.DataFrame(table, columns=['intervals', 'intervals_len', 'total_snp_count', 'interval_snp_count', 'asb_count'])
df['snp_per_1000k'] = df['interval_snp_count'] / df['intervals_len'] * 1000_000
df['%_asb'] = df['asb_count'] / df['interval_snp_count']
df

,intervals,intervals_len,total_snp_count,interval_snp_count,asb_count,snp_per_1000k,%_asb
0,peaks,2359071386,121611,121019,9973,51.299423,0.082409
1,TOPs,12085440,121611,3091,945,255.762306,0.305726
